# Charting the Grind

A project for the **Information Visualization course 2025/26** about understanding **personal study habits** through the visual exploration of a self-tracked study log, collected over roughly the last month of study sessions.

I started keeping this log because I wanted a clearer picture of what my study life actually looks like, beyond the vague feeling of "I studied a lot" or "this afternoon was not very productive". By looking at my sessions visually, I want to notice patterns I would probably miss otherwise: which topics take most of my time, when I seem to work better, which places help me focus and which kinds of activities feel more or less productive.

## Preparing the data

Before creating the visualizations, the raw study log needs to be cleaned and enriched. The CSV already contains the basic information about each study session, but some columns are still difficult to analyze directly. Primarily, start and finish times are written as text and the actual duration of each session is not explicitly available.  
In this phase, I **validate the dataset structure** and prepare it so that it can be used more easily for charts and interpretation.

First, I prepare the environment by importing the **libraries** I need and declaring all the **constants** used throughout the notebook.


In [1]:
import pandas as pd

In [2]:
# the input data csv file path
DATA_CSV = "data/study_sessions.csv"

# the columns that should be treated as categorical
CATEGORICAL_COLS = ["topic", "activity_type", "location"]

# the mapping of time of day bins for categorization
TIME_OF_DAY_CUTS = {
    "bins": [8, 13.5, 18.5, 24],
    "labels": ["morning", "afternoon", "evening"],
}

Then, I read the data from the CSV file into a **dataframe** and check its structure to make sure it contains all the expected columns, that they are in the right format, and that there are **no missing values** or other issues that could affect the analysis.

In [3]:
df = pd.read_csv(DATA_CSV)

In [4]:
# check the structure of the dataframe, especially the data types and missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 92 entries, 0 to 91
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   date                92 non-null     str  
 1   start               92 non-null     str  
 2   finish              92 non-null     str  
 3   topic               92 non-null     str  
 4   activity_type       92 non-null     str  
 5   productivity_score  92 non-null     int64
 6   location            92 non-null     str  
 7   description         92 non-null     str  
dtypes: int64(1), str(7)
memory usage: 5.9 KB


First, let's transform the **start** and **finish** times from text to datetime format and calculate the **duration** of each session in hours and minutes. This will allow us to analyze how long each session lasted and to create time-based visualizations later on.

In [5]:
# combine date + time into datetime columns
df["start_dt"] = pd.to_datetime(df["date"] + " " + df["start"], dayfirst=True)
df["finish_dt"] = pd.to_datetime(df["date"] + " " + df["finish"], dayfirst=True)

# calculate duration as a time delta
df["duration"] = df["finish_dt"] - df["start_dt"]

# convert duration to hours as a float
df["duration_hours"] = df["duration"].dt.total_seconds() / 3600

# convert duration to minutes as an integer
df["duration_minutes"] = (df["duration"].dt.total_seconds() / 60).astype(int)

# print the start, finish and duration columns to check the calculations
print(df[["start_dt", "finish_dt", "duration_hours", "duration_minutes"]])

              start_dt           finish_dt  duration_hours  duration_minutes
0  2026-04-13 10:00:00 2026-04-13 12:30:00             2.5               150
1  2026-04-14 09:30:00 2026-04-14 12:00:00             2.5               150
2  2026-04-14 15:30:00 2026-04-14 17:00:00             1.5                90
3  2026-04-14 17:00:00 2026-04-14 17:30:00             0.5                30
4  2026-04-14 18:30:00 2026-04-14 21:00:00             2.5               150
..                 ...                 ...             ...               ...
87 2026-05-17 11:00:00 2026-05-17 13:30:00             2.5               150
88 2026-05-17 15:00:00 2026-05-17 17:00:00             2.0               120
89 2026-05-18 09:30:00 2026-05-18 12:30:00             3.0               180
90 2026-05-18 16:00:00 2026-05-18 18:30:00             2.5               150
91 2026-05-18 21:00:00 2026-05-18 23:00:00             2.0               120

[92 rows x 4 columns]


Then, let's infer the **time of the day** for each session based on the start time. This will help us understand when I tend to study and whether there are any patterns in my productivity throughout the day.

In [6]:
# infer time of day from the start time using the defined bins and labels
df["time_of_day"] = pd.cut(
    df["start_dt"].dt.hour + df["start_dt"].dt.minute / 60,
    **TIME_OF_DAY_CUTS,
    right=False,
)

# print the start, finish and time of day columns to check the categorization
print(df[["start_dt", "finish_dt", "time_of_day"]])

              start_dt           finish_dt time_of_day
0  2026-04-13 10:00:00 2026-04-13 12:30:00     morning
1  2026-04-14 09:30:00 2026-04-14 12:00:00     morning
2  2026-04-14 15:30:00 2026-04-14 17:00:00   afternoon
3  2026-04-14 17:00:00 2026-04-14 17:30:00   afternoon
4  2026-04-14 18:30:00 2026-04-14 21:00:00     evening
..                 ...                 ...         ...
87 2026-05-17 11:00:00 2026-05-17 13:30:00     morning
88 2026-05-17 15:00:00 2026-05-17 17:00:00   afternoon
89 2026-05-18 09:30:00 2026-05-18 12:30:00     morning
90 2026-05-18 16:00:00 2026-05-18 18:30:00   afternoon
91 2026-05-18 21:00:00 2026-05-18 23:00:00     evening

[92 rows x 3 columns]


Finally, let's convert some of the columns that I know should be categorical to the appropriate data type for easier analysis.

In [7]:
# convert columns marked as categorical to category data type
df[CATEGORICAL_COLS] = df[CATEGORICAL_COLS].astype("category")

# print the unique values in categorical columns
for col in CATEGORICAL_COLS:
    unique_values = df[col].unique().tolist()
    unique_values.sort()
    print(f"Unique values in '{col}': {unique_values}")

Unique values in 'topic': ['Info Viz', 'KRKE', 'Open Science', 'SDL', 'SEDT', 'Work']
Unique values in 'activity_type': ['brainstorming', 'programming', 'reading', 'reviewing', 'writing']
Unique values in 'location': ['32', '34', '36', 'bigiavi', 'home', 'paleotti']


In [8]:
# final preflight check of the dataframe structure after all transformations
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 92 entries, 0 to 91
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype          
---  ------              --------------  -----          
 0   date                92 non-null     str            
 1   start               92 non-null     str            
 2   finish              92 non-null     str            
 3   topic               92 non-null     category       
 4   activity_type       92 non-null     category       
 5   productivity_score  92 non-null     int64          
 6   location            92 non-null     category       
 7   description         92 non-null     str            
 8   start_dt            92 non-null     datetime64[us] 
 9   finish_dt           92 non-null     datetime64[us] 
 10  duration            92 non-null     timedelta64[us]
 11  duration_hours      92 non-null     float64        
 12  duration_minutes    92 non-null     int64          
 13  time_of_day         92 non-null     category    